# Multi-Model LLM Baseline: Reappraisal Strategy Scoring

This notebook extends the original GLM LLM baseline to compare multiple OpenRouter-hosted LLMs on the same reappraisal strategy task.

Models tested by default:

- GLM 5.2: `z-ai/glm-5.2`
- DeepSeek V4 Pro: `deepseek/deepseek-v4-pro`
- Kimi K3: `moonshotai/kimi-k3`

The task remains multi-label and soft-label: each model returns eight independent scores in `[0, 1]`, one per reappraisal strategy. Scores are evaluated against human annotator vote fractions.

## 1. Imports And Configuration

In [ ]:
from pathlib import Path
import ast
import json
import os
import re
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.special import logit
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    hamming_loss,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split
from torch import nn
from tqdm.auto import tqdm

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
except ImportError:
    MultilabelStratifiedShuffleSplit = None

RANDOM_STATE = 42

# Locate the consolidated project from either the project root or a nested notebook folder.
def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'data' / 'annotate-lime.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')

PROJECT_DIR = Path(os.getenv('REAPPRAISAL_PROJECT_DIR') or find_project_root()).resolve()
DATA_DIR = Path(os.getenv('REAPPRAISAL_DATA_DIR') or PROJECT_DIR / 'data').resolve()
OUTPUT_DIR = Path(os.getenv('REAPPRAISAL_LLM_OUTPUT_DIR') or PROJECT_DIR / 'Code' / 'LLM' / 'outputs').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = os.getenv('OPENROUTER_API_KEY')
BASE_URL = 'https://openrouter.ai/api/v1'

# Keep this False until you have checked cost and row counts.
RUN_API = False

# Set to None for the full validation/test splits. Use a small integer for smoke tests.
N_ROWS_TO_SCORE = 10
SPLITS_TO_SCORE = ['validation', 'test']

TEMPERATURE = 0
MAX_TOKENS = 1000
REQUEST_SLEEP_SECONDS = 0.5
USE_RESPONSE_FORMAT = True

# These are reasoning-capable models, but this task only needs a short JSON answer.
# Minimal hidden reasoning reduces empty responses and unnecessary output-token cost.
OPENROUTER_EXTRA_BODY = {
    'provider': {'allow_fallbacks': False},
    'reasoning': {'effort': 'minimal', 'exclude': True},
}

# Human soft scores are binarized at this threshold only for secondary F1-style metrics.
LABEL_THRESHOLD = 0.6
MIN_EXAMPLE_SCORE = 0.8

MODEL_CONFIGS = [
    {
        'model_key': 'glm_5_2',
        'model_name': 'GLM 5.2',
        'model_id': 'z-ai/glm-5.2',
        'extra_request': {},
    },
    {
        'model_key': 'deepseek_v4_pro',
        'model_name': 'DeepSeek V4 Pro',
        'model_id': 'deepseek/deepseek-v4-pro',
        'extra_request': {},
    },
    {
        'model_key': 'kimi_k3',
        'model_name': 'Kimi K3',
        'model_id': 'moonshotai/kimi-k3',
        'extra_request': {},
    },
]

PROMPTING_STRATEGIES = [
    {
        'condition': 'zero_shot_codebook_only',
        'description': 'Zero-shot: codebook only',
        'n_examples_per_label': 0,
    },
    {
        'condition': 'few_shot_1_per_strategy',
        'description': 'Few-shot: one example per strategy',
        'n_examples_per_label': 1,
    },
    {
        'condition': 'few_shot_3_per_strategy',
        'description': 'Few-shot: three examples per strategy',
        'n_examples_per_label': 3,
    },
]

REAPPRAISAL_TECHNIQUES = ['[GM]', '[PR]', '[A]', '[EJ]', '[N]', '[UC]', '[V]', '[H]']
LABEL_COLS = [tech.strip('[]') for tech in REAPPRAISAL_TECHNIQUES]
LABEL_NAMES = {
    'GM': 'growth mindset',
    'PR': 'positive repurposing',
    'A': 'aberration',
    'EJ': 'external justification',
    'N': 'normalizing',
    'UC': 'unwarranted conclusions',
    'V': 'validation',
    'H': 'humor',
}

SCORE_COLS = [f'{label}_llm_score' for label in LABEL_COLS]
RAW_OUTPUT_PATH = OUTPUT_DIR / 'multi_model_prompting_raw_scores.csv'
COMPARISON_PATH = OUTPUT_DIR / 'multi_model_prompting_comparison.csv'
METRICS_PATH = OUTPUT_DIR / 'multi_model_prompting_metrics.json'
THRESHOLD_PATH = OUTPUT_DIR / 'multi_model_prompting_thresholds.json'

np.random.seed(RANDOM_STATE)

## 2. Load Texts And Build Human Soft Labels

Each annotation file is an annotator-by-item matrix. The target for each strategy is the fraction of valid annotators who marked that strategy present.

In [ ]:
texts = pd.read_csv(DATA_DIR / 'annotate-lime.csv')

soft_labels = pd.DataFrame()
vote_counts = pd.DataFrame()
positive_votes = pd.DataFrame()

for tech in REAPPRAISAL_TECHNIQUES:
    label = tech.strip('[]')
    ann = pd.read_csv(DATA_DIR / f'annotations-only-{tech}.csv', sep=';')
    ann = ann.apply(pd.to_numeric, errors='coerce')

    positive_votes[f'{label}_positive_votes'] = ann.sum(axis=0).values
    vote_counts[f'{label}_n_votes'] = ann.count(axis=0).values
    soft_labels[label] = ann.mean(axis=0).values

dataset = pd.concat(
    [
        texts[['key', 'SITUATION', 'REAPPRAISAL']].reset_index(drop=True),
        soft_labels.reset_index(drop=True),
        positive_votes.reset_index(drop=True),
        vote_counts.reset_index(drop=True),
    ],
    axis=1,
)

dataset.insert(0, 'question_code', [f'Q{i + 1}' for i in range(len(dataset))])
binary_labels = (dataset[LABEL_COLS] >= LABEL_THRESHOLD).astype(int)

print('Dataset shape:', dataset.shape)
display(dataset.head())

## 3. Train / Validation / Test Split

The models are not trained. The training split is used only for few-shot examples. The validation split is used for threshold tuning. The test split is used for final evaluation.

In [ ]:
def multilabel_split(df, y_binary, test_size, random_state):
    if MultilabelStratifiedShuffleSplit is not None:
        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=random_state,
        )
        left_idx, right_idx = next(splitter.split(df, y_binary))
        return df.iloc[left_idx].reset_index(drop=True), df.iloc[right_idx].reset_index(drop=True)

    return train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )


train_df, temp_df = multilabel_split(dataset, binary_labels, test_size=0.30, random_state=RANDOM_STATE)
temp_binary = (temp_df[LABEL_COLS] >= LABEL_THRESHOLD).astype(int)
val_df, test_df = multilabel_split(temp_df, temp_binary, test_size=0.50, random_state=RANDOM_STATE)

split_frames = {
    'train': train_df.reset_index(drop=True),
    'validation': val_df.reset_index(drop=True),
    'test': test_df.reset_index(drop=True),
}

for name, split_df in split_frames.items():
    print(f'\n{name}: {len(split_df)} rows')
    display((split_df[LABEL_COLS] >= LABEL_THRESHOLD).astype(int).sum().to_frame('majority_positive_items').T)

## 4. Prompt Construction

Every model receives the same codebook, output schema, examples, temperature, and row order. The only planned difference is the model ID.

In [ ]:
CODEBOOK = """
You are labeling interpersonal reappraisals for psychological research.

Return eight independent soft scores in [0, 1]. Each score means:
P(a trained human annotator would mark this strategy as present | situation, reappraisal).

The strategies are not mutually exclusive. Do not force the scores to sum to 1.

Labels:
GM: growth mindset, understanding the situation as a learning opportunity.
PR: positive repurposing, reevaluating the intentions of the other person(s) involved.
A: aberration, thinking of the situation as an exception rather than the norm.
EJ: external justification, understanding that the cause of the situation might lie outside a person's control.
N: normalizing, understanding that the situation is normal and advantageous.
UC: unwarranted conclusions, questioning a person's conclusions from the situation.
V: validation, validating a person's perception of the situation.
H: humor, giving the situation a humorous twist.

Respond only with valid JSON using exactly these keys:
{\"GM\": 0.0, \"PR\": 0.0, \"A\": 0.0, \"EJ\": 0.0, \"N\": 0.0, \"UC\": 0.0, \"V\": 0.0, \"H\": 0.0}
""".strip()


def score_dict(row, decimals=2):
    return {label: round(float(row[label]), decimals) for label in LABEL_COLS}


def build_few_shot_examples(train_df, n_per_label, min_score=MIN_EXAMPLE_SCORE):
    if n_per_label <= 0:
        return ''

    examples = []
    used_question_codes = set()

    for label in LABEL_COLS:
        candidates = train_df[train_df[label] >= min_score].sort_values(label, ascending=False)
        if len(candidates) < n_per_label:
            candidates = train_df.sort_values(label, ascending=False)

        selected = []
        for _, row in candidates.iterrows():
            if row['question_code'] in used_question_codes:
                continue
            selected.append(row)
            used_question_codes.add(row['question_code'])
            if len(selected) == n_per_label:
                break

        for row in selected:
            examples.append(
                'SITUATION: ' + str(row['SITUATION']) + '\n'
                'REAPPRAISAL: ' + str(row['REAPPRAISAL']) + '\n'
                'HUMAN_SOFT_SCORES: ' + json.dumps(score_dict(row))
            )

    return '\n\n'.join(examples)


FEW_SHOT_BLOCKS = {
    cfg['condition']: build_few_shot_examples(train_df, cfg['n_examples_per_label'])
    for cfg in PROMPTING_STRATEGIES
}

for cfg in PROMPTING_STRATEGIES:
    condition = cfg['condition']
    print(condition, 'examples:', FEW_SHOT_BLOCKS[condition].count('HUMAN_SOFT_SCORES'))


def construct_message(row, strategy_config):
    condition = strategy_config['condition']
    few_shot_block = FEW_SHOT_BLOCKS[condition]

    parts = [CODEBOOK]
    if few_shot_block:
        parts.append('Here are examples with human annotator vote fractions from the training split:\n' + few_shot_block)
    parts.append(
        'Now score this item. Return only valid JSON.\n'
        f'SITUATION: {row["SITUATION"]}\n'
        f'REAPPRAISAL: {row["REAPPRAISAL"]}'
    )
    return '\n\n'.join(parts)


print(construct_message(test_df.iloc[0], PROMPTING_STRATEGIES[0])[:1800])

## 5. API Call And Response Parsing

Results are cached by model, condition, split, and question code so interrupted runs can resume safely.

In [ ]:
def get_client():
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise ImportError(
            'Install the OpenAI Python package before RUN_API=True, for example: pip install openai'
        ) from exc

    if not API_KEY:
        raise ValueError('Set OPENROUTER_API_KEY before RUN_API=True.')
    return OpenAI(base_url=BASE_URL, api_key=API_KEY)


def parse_llm_scores(raw_text):
    text = '' if raw_text is None else str(raw_text).strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.IGNORECASE | re.DOTALL).strip()
    if not text:
        raise ValueError('The LLM returned an empty response. Increase MAX_TOKENS or try another model.')

    json_match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if json_match:
        parsed = json.loads(json_match.group(0))
        missing = [label for label in LABEL_COLS if label not in parsed]
        if missing:
            raise ValueError(f'LLM JSON is missing labels {missing}. Raw response: {text[:1000]}')
        return {label: float(np.clip(float(parsed[label]), 0.0, 1.0)) for label in LABEL_COLS}

    # Backward-compatible fallback for models that return [1, 4, 7]-style labels.
    try:
        maybe_list = ast.literal_eval(text)
    except (SyntaxError, ValueError) as exc:
        raise ValueError(f'Could not parse LLM response as JSON or label list. Raw response: {text[:1000]!r}') from exc

    number_to_label = {1: 'GM', 2: 'PR', 3: 'A', 4: 'EJ', 5: 'N', 6: 'UC', 7: 'V', 8: 'H'}
    labels = {number_to_label[int(x)] for x in maybe_list}
    return {label: float(label in labels) for label in LABEL_COLS}


def call_llm(client, row, model_config, strategy_config):
    request = {
        'model': model_config['model_id'],
        'messages': [
            {'role': 'system', 'content': 'You are a careful psychological text annotation assistant.'},
            {'role': 'user', 'content': construct_message(row, strategy_config)},
        ],
        'max_tokens': MAX_TOKENS,
        'temperature': TEMPERATURE,
    }
    if USE_RESPONSE_FORMAT:
        request['response_format'] = {'type': 'json_object'}
    if OPENROUTER_EXTRA_BODY:
        request['extra_body'] = OPENROUTER_EXTRA_BODY
    request.update(model_config.get('extra_request', {}))

    completion = client.chat.completions.create(**request)
    message = completion.choices[0].message
    raw_response = message.content
    finish_reason = getattr(completion.choices[0], 'finish_reason', None)

    usage = getattr(completion, 'usage', None)
    usage_payload = {}
    if usage is not None:
        usage_payload = {
            'prompt_tokens': getattr(usage, 'prompt_tokens', None),
            'completion_tokens': getattr(usage, 'completion_tokens', None),
            'total_tokens': getattr(usage, 'total_tokens', None),
        }

    return raw_response, getattr(completion, 'model', None), finish_reason, usage_payload


def load_cached_scores(path=RAW_OUTPUT_PATH):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def save_cached_scores(df, path=RAW_OUTPUT_PATH):
    df.to_csv(path, index=False)
    print(f'Saved cache: {path}')

## 6. Run Or Load LLM Scores

In [ ]:
def score_model_condition_split(model_config, strategy_config, split_name, split_df, client, cache_df):
    model_key = model_config['model_key']
    condition = strategy_config['condition']
    rows_to_score = split_df.copy()
    if N_ROWS_TO_SCORE is not None:
        rows_to_score = rows_to_score.head(N_ROWS_TO_SCORE)

    cached_ids = set()
    required_cols = {'model_key', 'condition', 'split', 'question_code'}
    if not cache_df.empty and required_cols.issubset(cache_df.columns):
        mask = (
            (cache_df['model_key'] == model_key)
            & (cache_df['condition'] == condition)
            & (cache_df['split'] == split_name)
        )
        cached_ids = set(cache_df.loc[mask, 'question_code'])

    new_rows = []
    desc = f'{model_key} | {condition} | {split_name}'
    for _, row in tqdm(rows_to_score.iterrows(), total=len(rows_to_score), desc=desc):
        if row['question_code'] in cached_ids:
            continue

        raw_response, actual_model, finish_reason, usage_payload = call_llm(client, row, model_config, strategy_config)
        result = {
            'model_key': model_key,
            'model_name': model_config['model_name'],
            'model_id': model_config['model_id'],
            'actual_model': actual_model,
            'condition': condition,
            'description': strategy_config['description'],
            'n_examples_per_label': strategy_config['n_examples_per_label'],
            'split': split_name,
            'question_code': row['question_code'],
            'key': row['key'],
            'raw_response': raw_response,
            'finish_reason': finish_reason,
            **usage_payload,
        }

        try:
            parsed_scores = parse_llm_scores(raw_response)
        except ValueError:
            failed_path = OUTPUT_DIR / f'failed_response_{model_key}_{condition}_{split_name}_{row["question_code"]}.txt'
            failed_payload = {
                'model_key': model_key,
                'model_id': model_config['model_id'],
                'condition': condition,
                'split': split_name,
                'question_code': row['question_code'],
                'finish_reason': finish_reason,
                'usage': usage_payload,
                'raw_response': raw_response,
            }
            failed_path.write_text(json.dumps(failed_payload, indent=2))
            print(f'Failed to parse response. Raw response saved to: {failed_path}')
            raise

        result.update({f'{label}_llm_score': parsed_scores[label] for label in LABEL_COLS})
        new_rows.append(result)

        updated_cache = pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)
        save_cached_scores(updated_cache)
        time.sleep(REQUEST_SLEEP_SECONDS)

    return pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)


cache_df = load_cached_scores()

if RUN_API:
    client = get_client()
    for model_config in MODEL_CONFIGS:
        for strategy_config in PROMPTING_STRATEGIES:
            for split_name in SPLITS_TO_SCORE:
                cache_df = score_model_condition_split(
                    model_config,
                    strategy_config,
                    split_name,
                    split_frames[split_name],
                    client,
                    cache_df,
                )
else:
    print('RUN_API is False. Set OPENROUTER_API_KEY, set RUN_API=True, and rerun this cell to score rows.')

llm_scores = load_cached_scores()
print('Cached LLM score rows:', len(llm_scores))
display(llm_scores.head())

## 7. Evaluation Helpers

In [ ]:
def get_arrays(eval_df):
    y_true = eval_df[LABEL_COLS].to_numpy(dtype=float)
    y_score = eval_df[SCORE_COLS].to_numpy(dtype=float)
    return y_true, y_score


def safe_corr(fn, y_true, y_score):
    if np.std(y_true) == 0 or np.std(y_score) == 0:
        return np.nan
    return fn(y_true, y_score).statistic


def soft_label_metrics(y_true, y_score, label_cols):
    rows = []
    for i, label in enumerate(label_cols):
        yt = y_true[:, i]
        ys = y_score[:, i]
        rows.append({
            'label': label,
            'strategy': LABEL_NAMES[label],
            'mae': mean_absolute_error(yt, ys),
            'rmse': mean_squared_error(yt, ys) ** 0.5,
            'pearson': safe_corr(pearsonr, yt, ys),
            'spearman': safe_corr(spearmanr, yt, ys),
            'mean_true': yt.mean(),
            'mean_pred': ys.mean(),
        })
    return pd.DataFrame(rows)


def soft_loss_summary(y_true, y_score):
    clipped_scores = np.clip(y_score, 1e-6, 1 - 1e-6)
    logits = logit(clipped_scores)

    y_true_t = torch.tensor(y_true, dtype=torch.float32)
    y_score_t = torch.tensor(y_score, dtype=torch.float32)
    logits_t = torch.tensor(logits, dtype=torch.float32)

    return {
        'bce_with_logits_loss': nn.BCEWithLogitsLoss()(logits_t, y_true_t).item(),
        'mse_loss': nn.MSELoss()(y_score_t, y_true_t).item(),
    }


def tune_thresholds(y_true_soft, y_score, label_cols, label_threshold=LABEL_THRESHOLD, grid=None):
    if grid is None:
        grid = np.arange(0.05, 0.96, 0.05)

    y_true_binary = (y_true_soft >= label_threshold).astype(int)
    thresholds = {}
    rows = []

    for i, label in enumerate(label_cols):
        best_threshold = 0.5
        best_f1 = -1.0

        for threshold in grid:
            pred = (y_score[:, i] >= threshold).astype(int)
            score = f1_score(y_true_binary[:, i], pred, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_threshold = threshold

        thresholds[label] = float(best_threshold)
        rows.append({'label': label, 'threshold': best_threshold, 'validation_f1': best_f1})

    return thresholds, pd.DataFrame(rows)


def apply_thresholds(scores, thresholds, label_cols):
    threshold_array = np.array([thresholds[label] for label in label_cols])
    return (scores >= threshold_array).astype(int)


def evaluate_multilabel(name, y_true_soft, scores, thresholds, label_cols):
    y_true = (y_true_soft >= LABEL_THRESHOLD).astype(int)
    y_pred = apply_thresholds(scores, thresholds, label_cols)

    metrics = {
        'split': name,
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'samples_f1': f1_score(y_true, y_pred, average='samples', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
    }

    ap_rows = []
    count_rows = []
    for i, label in enumerate(label_cols):
        if len(np.unique(y_true[:, i])) < 2:
            ap = np.nan
        else:
            ap = average_precision_score(y_true[:, i], scores[:, i])
        ap_rows.append({'label': label, 'average_precision': ap})
        count_rows.append({
            'label': label,
            'true_positive_items': int(y_true[:, i].sum()),
            'predicted_positive_items': int(y_pred[:, i].sum()),
        })

    report = classification_report(y_true, y_pred, target_names=label_cols, zero_division=0, output_dict=True)
    return metrics, pd.DataFrame(ap_rows), pd.DataFrame(count_rows), y_pred, report


def merge_model_condition_split_scores(model_key, condition, split_name, split_df, llm_scores):
    if llm_scores.empty or 'model_key' not in llm_scores.columns:
        return pd.DataFrame()

    split_scores = llm_scores[
        (llm_scores['model_key'] == model_key)
        & (llm_scores['condition'] == condition)
        & (llm_scores['split'] == split_name)
    ].copy()
    if split_scores.empty:
        return pd.DataFrame()

    keep_cols = [
        'model_key',
        'model_name',
        'model_id',
        'actual_model',
        'condition',
        'description',
        'question_code',
        'key',
        'raw_response',
    ] + SCORE_COLS
    optional_cols = [col for col in ['prompt_tokens', 'completion_tokens', 'total_tokens'] if col in split_scores.columns]
    return split_df.merge(split_scores[keep_cols + optional_cols], on=['question_code', 'key'], how='inner')

## 8. Evaluate All Models And Prompting Strategies

In [ ]:
def evaluate_model_condition(model_config, strategy_config, llm_scores):
    model_key = model_config['model_key']
    condition = strategy_config['condition']
    description = strategy_config['description']

    val_eval_df = merge_model_condition_split_scores(model_key, condition, 'validation', val_df, llm_scores)
    test_eval_df = merge_model_condition_split_scores(model_key, condition, 'test', test_df, llm_scores)

    print(f'\n=== {model_config["model_name"]} | {description} ===')
    print('Validation rows with LLM scores:', len(val_eval_df))
    print('Test rows with LLM scores:', len(test_eval_df))

    if test_eval_df.empty:
        print('No test scores yet for this model/condition.')
        return None

    test_labels, test_scores = get_arrays(test_eval_df)
    test_soft = soft_label_metrics(test_labels, test_scores, LABEL_COLS)
    test_losses = soft_loss_summary(test_labels, test_scores)

    fixed_thresholds = {label: 0.5 for label in LABEL_COLS}
    fixed_metrics, fixed_ap, fixed_counts, fixed_pred, fixed_report = evaluate_multilabel(
        f'{model_key} {condition} test fixed 0.5', test_labels, test_scores, fixed_thresholds, LABEL_COLS
    )

    if not val_eval_df.empty:
        val_labels, val_scores = get_arrays(val_eval_df)
        thresholds, threshold_df = tune_thresholds(val_labels, val_scores, LABEL_COLS)
    else:
        thresholds = fixed_thresholds
        threshold_df = pd.DataFrame({'label': LABEL_COLS, 'threshold': [0.5] * len(LABEL_COLS), 'validation_f1': np.nan})

    tuned_metrics, tuned_ap, tuned_counts, tuned_pred, tuned_report = evaluate_multilabel(
        f'{model_key} {condition} test tuned', test_labels, test_scores, thresholds, LABEL_COLS
    )

    pred_df = test_eval_df[
        ['model_key', 'model_name', 'model_id', 'condition', 'description', 'question_code', 'key', 'SITUATION', 'REAPPRAISAL', 'raw_response']
    ].copy()
    for i, label in enumerate(LABEL_COLS):
        pred_df[f'{label}_true_score'] = test_labels[:, i]
        pred_df[f'{label}_llm_score'] = test_scores[:, i]
        pred_df[f'{label}_true_binary'] = (test_labels[:, i] >= LABEL_THRESHOLD).astype(int)
        pred_df[f'{label}_pred_binary_fixed_0_5'] = fixed_pred[:, i]
        pred_df[f'{label}_pred_binary_tuned'] = tuned_pred[:, i]

    prediction_path = OUTPUT_DIR / f'{model_key}_{condition}_test_predictions.csv'
    pred_df.to_csv(prediction_path, index=False)

    summary = {
        'model_key': model_key,
        'model_name': model_config['model_name'],
        'model_id': model_config['model_id'],
        'condition': condition,
        'description': description,
        'n_examples_per_label': strategy_config['n_examples_per_label'],
        'n_validation_rows': len(val_eval_df),
        'n_test_rows': len(test_eval_df),
        'macro_mae': float(test_soft['mae'].mean()),
        'macro_rmse': float(test_soft['rmse'].mean()),
        'mean_pearson': float(test_soft['pearson'].mean(skipna=True)),
        'mean_spearman': float(test_soft['spearman'].mean(skipna=True)),
        'bce_with_logits_loss': test_losses['bce_with_logits_loss'],
        'mse_loss': test_losses['mse_loss'],
        'fixed_micro_f1': fixed_metrics['micro_f1'],
        'fixed_macro_f1': fixed_metrics['macro_f1'],
        'fixed_samples_f1': fixed_metrics['samples_f1'],
        'fixed_hamming_loss': fixed_metrics['hamming_loss'],
        'tuned_micro_f1': tuned_metrics['micro_f1'],
        'tuned_macro_f1': tuned_metrics['macro_f1'],
        'tuned_samples_f1': tuned_metrics['samples_f1'],
        'tuned_hamming_loss': tuned_metrics['hamming_loss'],
        'prediction_path': str(prediction_path),
    }

    return {
        'summary': summary,
        'thresholds': thresholds,
        'threshold_df': threshold_df,
        'test_soft_metrics': test_soft,
        'test_losses': test_losses,
        'fixed_metrics': fixed_metrics,
        'tuned_metrics': tuned_metrics,
        'fixed_average_precision': fixed_ap,
        'tuned_average_precision': tuned_ap,
        'fixed_prediction_counts': fixed_counts,
        'tuned_prediction_counts': tuned_counts,
        'fixed_classification_report': fixed_report,
        'tuned_classification_report': tuned_report,
    }


condition_results = {}
summary_rows = []
threshold_payload = {}
metrics_payload = {
    'config': {
        'base_url': BASE_URL,
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'label_threshold': LABEL_THRESHOLD,
        'min_example_score': MIN_EXAMPLE_SCORE,
        'n_rows_to_score': N_ROWS_TO_SCORE,
        'model_configs': MODEL_CONFIGS,
        'openrouter_extra_body': OPENROUTER_EXTRA_BODY,
        'prompting_strategies': PROMPTING_STRATEGIES,
    },
    'results': {},
}

for model_config in MODEL_CONFIGS:
    for strategy_config in PROMPTING_STRATEGIES:
        result = evaluate_model_condition(model_config, strategy_config, llm_scores)
        if result is None:
            continue

        result_key = f'{model_config["model_key"]}__{strategy_config["condition"]}'
        condition_results[result_key] = result
        summary_rows.append(result['summary'])
        threshold_payload[result_key] = result['thresholds']
        metrics_payload['results'][result_key] = {
            'summary': result['summary'],
            'thresholds': result['thresholds'],
            'test_soft_metrics': result['test_soft_metrics'].to_dict(orient='records'),
            'test_losses': result['test_losses'],
            'fixed_metrics': result['fixed_metrics'],
            'tuned_metrics': result['tuned_metrics'],
            'fixed_average_precision': result['fixed_average_precision'].to_dict(orient='records'),
            'tuned_average_precision': result['tuned_average_precision'].to_dict(orient='records'),
            'fixed_prediction_counts': result['fixed_prediction_counts'].to_dict(orient='records'),
            'tuned_prediction_counts': result['tuned_prediction_counts'].to_dict(orient='records'),
            'fixed_classification_report': result['fixed_classification_report'],
            'tuned_classification_report': result['tuned_classification_report'],
        }

comparison_df = pd.DataFrame(summary_rows)
if comparison_df.empty:
    print('No scored model/condition pairs yet. Run the API scoring cell first.')
else:
    comparison_df.to_csv(COMPARISON_PATH, index=False)
    THRESHOLD_PATH.write_text(json.dumps(threshold_payload, indent=2))
    METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2))
    display(comparison_df.sort_values(['macro_mae', 'macro_rmse']).round(3))
    print('Saved:')
    print(COMPARISON_PATH)
    print(METRICS_PATH)
    print(THRESHOLD_PATH)

## 9. Inspect One Model/Condition

In [ ]:
def result_key(model_key, condition):
    return f'{model_key}__{condition}'


def show_condition_details(model_key, condition):
    result = condition_results[result_key(model_key, condition)]
    print(result_key(model_key, condition))
    print('\nThresholds')
    display(result['threshold_df'].round(3))
    print('\nSoft-label metrics')
    display(result['test_soft_metrics'].round(3))
    print('\nFixed 0.5 prediction counts')
    display(result['fixed_prediction_counts'])
    print('\nTuned prediction counts')
    display(result['tuned_prediction_counts'])


# Examples after evaluation:
# show_condition_details('glm_5_2', 'zero_shot_codebook_only')
# show_condition_details('deepseek_v4_pro', 'few_shot_1_per_strategy')
# show_condition_details('kimi_k3', 'few_shot_3_per_strategy')

## 10. Error Analysis Helper

In [ ]:
def inspect_label(model_key, condition, label, n=10, false_positive=True, threshold_mode='tuned'):
    prediction_path = OUTPUT_DIR / f'{model_key}_{condition}_test_predictions.csv'
    if not prediction_path.exists():
        raise ValueError(f'Run evaluation first. Missing: {prediction_path}')

    pred_df = pd.read_csv(prediction_path)
    score_col = f'{label}_llm_score'
    true_col = f'{label}_true_binary'
    pred_col = f'{label}_pred_binary_{threshold_mode}'

    if false_positive:
        subset = pred_df[(pred_df[true_col] == 0) & (pred_df[pred_col] == 1)].sort_values(score_col, ascending=False)
    else:
        subset = pred_df[(pred_df[true_col] == 1) & (pred_df[pred_col] == 0)].sort_values(score_col, ascending=True)

    return subset[
        [
            'model_key',
            'condition',
            'question_code',
            'key',
            'SITUATION',
            'REAPPRAISAL',
            f'{label}_true_score',
            score_col,
            true_col,
            pred_col,
            'raw_response',
        ]
    ].head(n)


# Examples after evaluation:
# inspect_label('glm_5_2', 'few_shot_1_per_strategy', 'EJ', false_positive=True, threshold_mode='fixed_0_5')
# inspect_label('kimi_k3', 'zero_shot_codebook_only', 'PR', false_positive=False, threshold_mode='tuned')

## 11. Reporting Checklist

For a real model comparison, report:

1. Soft-label metrics first: MAE, RMSE, Pearson, Spearman.
2. The comparison table across model and prompting condition.
3. Fixed `0.5` threshold F1 as the cleaner secondary binary view.
4. Validation-tuned F1 only after the full validation split has been scored.
5. Prediction counts by label, especially overprediction of `EJ` and `V`, and rare-label behavior for `A` and `H`.
6. Cost, latency, parse failures, retries, and actual routed model names from the cache.

For a publication-quality comparison, set `N_ROWS_TO_SCORE = None`, keep the same prompts and splits for every model, and freeze the final model IDs in the methods section.